In [ ]:
# ======================================================
# Part 1: 项目初始化与加载资源
# ======================================================

# --- 1. 安装所有必需的库 ---
%pip install transformers datasets peft accelerate bitsandbytes sentencepiece trl
%pip install jieba

In [ ]:
# --- 2. 导入所有需要的模块 ---
import torch
import random
import jieba
import jieba.posseg as pseg
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

print("所有依赖已安装并导入成功！")

In [ ]:
# --- 3. 从本地文件加载并准备数据集 ---

# 数据集信息
local_poem_file = "p5-4.txt" 
num_samples = 2000          # 选取部分数据快速实验

def load_poems_from_txt(file_path):
    """从每行一首诗的 txt 文件中读取数据"""
    with open(file_path, 'r', encoding='utf-8') as f:
        # 读取所有行，并去除每行末尾的换行符
        poems = [line.strip() for line in f.readlines()]
    
    return Dataset.from_dict({"content": poems})

print(f"正在从本地文件 '{local_poem_file}' 加载数据集...")

poem_dataset_full = load_poems_from_txt(local_poem_file)

# poem_dataset = poem_dataset_full.select(range(num_samples))
poem_dataset = poem_dataset_full

print(f"成功加载 {len(poem_dataset)} 首五言绝句。")
print("\n数据集示例:")

print(poem_dataset[0]['content'])

In [ ]:
# --- 4. 定义模型、Tokenizer ---
model_id = "LiquidAI/LFM2-1.2B"
model_path = "./LFM2-1.2B"

# --- 加载 LFM2 的 Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer '{model_id}' 加载成功。")
print(f"Tokenizer 的 pad_token ID: {tokenizer.pad_token_id}")
print(f"Tokenizer 的 eos_token ID: {tokenizer.eos_token_id}")


# --- 5. 编写基于关键词提取的预处理函数 (适配 LFM2) ---

def preprocess_function(example):
    """
    将一首诗转换成 LFM2 的 ChatTemplate 格式的对话样本
    """
    poem_text = example['content']
    
    # 1. 清理诗歌文本中的标点和空格
    poem_clean = poem_text.replace('，', '').replace('。', '').strip()
    
    # 2. 使用 jieba 提取名词作为关键词
    try:
        words = pseg.cut(poem_clean)
        nouns = [word for word, flag in words if flag.startswith('n')]
    except Exception:
        # 如果 pseg 出错，回退到简单分词
        words = jieba.cut(poem_clean)
        nouns = list(words) # 将所有词都作为候选关键词

    
    # 3. 构造指令
    if nouns:
        # 随机选择1-3个名词作为关键词
        num_keywords = min(len(nouns), 3)
        selected_keywords = random.sample(nouns, num_keywords)
        keyword = "、".join(selected_keywords)
        instruction = f"请为我创作一首关于“{keyword}”的五言绝句。"
    else:
        # 如果没有提取到名词，就用一个通用指令
        instruction = "请为我创作一首经典的五言绝句。"

    # 4. 应用 Chat Template (ChatML 模板)
    example["messages"] = [
        {"role": "system", "content": "You are a helpful assistant trained by Liquid AI."},
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": poem_text}
    ]
    return example

# --- 6. 对整个数据集应用预处理函数 ---
print("\n正在对诗歌数据集进行预处理...")
formatted_dataset = poem_dataset.map(preprocess_function, remove_columns=poem_dataset.column_names)

print("--- 数据集已转换为 messages 格式 ---")
print("一个样本示例:")
print(formatted_dataset[0]['messages'])

In [ ]:
# ======================================================
# Part 2: 加载模型并应用 QLoRA
# ======================================================

# --- 1. 配置 QLoRA (4-bit 量化) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                # 在4-bit精度下加载模型
    bnb_4bit_use_double_quant=True,   # 使用双量化，进一步节省显存
    bnb_4bit_quant_type="nf4",        # 使用 nf4 (Normal Float 4) 量化类型
    bnb_4bit_compute_dtype=torch.bfloat16 # 在计算时，使用 bfloat16 来保持精度和性能
)
print("--- QLoRA (BitsAndBytes) 配置创建成功 ---")

# --- 2. 加载 4-bit 量化后的基础模型 ---
print(f"--- 正在从本地路径 '{model_path}' 加载 4-bit 量化模型 ---")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto", 
    trust_remote_code=True,
)
print("--- 基础模型加载成功 ---")

# --- 3. 准备模型进行 LoRA 微调 ---
model = prepare_model_for_kbit_training(model)
print("--- 模型已准备好进行 k-bit 训练 ---")

# --- 4. 定义 LoRA 配置 ---
# LoraConfig 定义了 LoRA 适配器的所有超参数
lora_config = LoraConfig(
    r=16,                             # LoRA 的秩
    lora_alpha=32,                    # LoRA 的 alpha 缩放因子
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # 指定要应用 LoRA 的层。
    lora_dropout=0.05,                # LoRA 层的 dropout 概率
    bias="none",                      # 不训练 bias
    task_type="CAUSAL_LM"             # 任务类型为 CLM
)
print("--- LoRA 配置创建成功 ---")


# --- 5. 将 LoRA注入到模型中 ---
# get_peft_model 会根据 lora_config 自动找到目标层并应用 LoRA
lora_model = get_peft_model(model, lora_config)
print("--- LoRA 适配器已成功注入模型 ---")


# --- 6. 打印模型的可训练参数信息 ---
print("\n" + "="*50)
lora_model.print_trainable_parameters()
print("="*50)

# print(lora_model)

In [ ]:
# ======================================================
# Part 3: 配置 Trainer 并开始微调
# ======================================================

# --- 1. 配置训练参数  ---
# TrainingArguments 类包含了所有可以自定义的训练选项
training_args = SFTConfig(
    # --- 核心参数 ---
    output_dir="./outputs/lfm2_poet_lora",   # 训练输出（checkpoint等）的保存路径
    num_train_epochs=3,                     # 总共训练 3 个轮次
    per_device_train_batch_size=2,          # 每个 GPU 上的训练 batch size
    gradient_accumulation_steps=8,          # 梯度累积步数，有效 batch size = 2 * 8 = 16

    # --- 优化器与学习率调度器 ---
    optim="paged_adamw_8bit",               # 使用 QLoRA 推荐的节省显存的优化器
    learning_rate=2e-4,                     # 学习率
    lr_scheduler_type="cosine",             # 使用余弦学习率衰减
    warmup_ratio=0.03,                      # 预热步数的比例

    # --- 硬件与精度 ---
    fp16=False,                             # 不启用 fp16
    bf16=True,                              # 启用 bfloat16 混合精度训练

    # --- 日志与保存 ---
    logging_steps=10,                       # 每 10 步记录一次日志
    save_strategy="steps",                  # 按步数保存模型
    save_steps=50,                          # 每 50 步保存一次 checkpoint
    save_total_limit=3,                     # 最多保留 3 个 checkpoint

    # --- 其他 ---
    push_to_hub=False,                      # 不将模型推送到 Hugging Face Hub
    report_to="none",                       # 不使用 wandb 或 tensorboard 等进行报告
    packing=True,                           # 启用 example packing 加速训练
)

print("--- 训练参数配置完成 ---")


# --- 2. 创建 Trainer 实例 ---
trainer = SFTTrainer(
    model=lora_model,                # 传入我们加载的 4-bit 基础模型
    args=training_args,              # 传入训练配置
    train_dataset=formatted_dataset, # 传入我们处理好的 messages 格式数据集 
)

print("--- Trainer 实例创建成功 ---")


# --- 3. 开始微调！ ---
print("\n" + "="*50)
print("开始 LoRA 微调模型")
print("="*50)

trainer.train()

print("\n" + "="*50)
print("微调完成！")
print("="*50)


# --- 4. 保存最终的 LoRA Adapter ---
# 训练完成后，再手动保存一次最终的模型
final_model_path = "./outputs/lfm2_poet_lora_final"
lora_model.save_pretrained(final_model_path)
print(f"\n 最终的 LoRA Adapter 已保存到: {final_model_path}")

In [ ]:
# ======================================================
# Part 4: 推理与效果展示
# ======================================================

# --- 1. 定义本地模型和适配器路径 ---
# 基础模型路径
model_path = "./LFM2-1.2B" 
# 你训练好的 LoRA 适配器路径
adapter_path = "./outputs/lfm2_poet_lora_final"

print("--- 准备加载模型用于对比测试 ---")

# --- 2. 加载未经微调的基座模型 ---
print("\n--- 正在加载微调前的基座模型 ---")

# 同样使用 4-bit 量化加载，以节省显存
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
print("基座模型加载成功！")


# --- 3. 加载微调后的 LoRA 模型 ---
print("\n--- 正在加载微调后的 LoRA 模型 ---")
# 首先加载基础模型
lora_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
# 然后加载 LoRA 适配器并与基础模型合并
# 这会在基础模型之上，“附加”上你训练好的 LoRA 权重
lora_model = PeftModel.from_pretrained(lora_model, adapter_path)
lora_model = lora_model.merge_and_unload() 

lora_model.eval() # 切换到评估模式
print("LoRA 模型加载并合并成功！")


# --- 4. 编写统一的生成函数 ---
def generate_poem(model, instruction, tokenizer):
    """使用给定的模型和指令生成文本"""
    
    # 准备聊天模板
    messages = [{"role": "user", "content": instruction}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 编码输入
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 使用 torch.no_grad() 进行推理，以节省资源
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=60,         # 五言绝句大约20个字，60个 token 足够了
            do_sample=True,            # 开启采样，让生成不那么死板
            temperature=0.3,           # LFM2 推荐的生成参数
            min_p=0.15,
            repetition_penalty=1.05    # LFM2 推荐的生成参数
        )
    
    # 解码输出
    response_ids = outputs[0][inputs['input_ids'].shape[1]:] # 只取生成的部分
    response = tokenizer.decode(response_ids, skip_special_tokens=True)
    
    return response


# --- 5. 进行对比测试 ---
instruction = "请为我创作一首关于“明月”的五言绝句。"

print("\n" + "="*50)
print(f"Prompt: {instruction}")
print("="*50)

print("\n--- LFM2 (微调前) ---")
base_model_output = generate_poem(base_model, instruction, tokenizer)
print(base_model_output)

print("\n\n--- LFM2 (微调后) ---")
lora_model_output = generate_poem(lora_model, instruction, tokenizer)
print(lora_model_output)
print("\n" + "="*50)